# 04 — Experimentación del módulo ML

Entrenamiento y comparación de cinco modelos supervisados sobre `crop_recommendation.csv` para clasificar el cultivo recomendado (22 clases, 2,200 filas, 7 features numéricos). El modelo ganador por F1 macro se reimplementa desde cero en `src/ml/models/champion.py` (sección final).

**Modelos evaluados:**
- KNN con k optimizado por k-fold
- Naive Bayes Gaussiano
- Decision Tree con profundidad optimizada
- SVM kernel RBF
- MLP 1–2 capas ocultas

**Protocolo:** mismo split train/test estratificado (80/20, seed=42), mismas métricas, k-fold (5 folds) para ajuste de hiperparámetros donde aplica. Métricas reportadas: accuracy, F1 macro, tiempo de entrenamiento.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.ml.preprocess import (
    load_crop_dataset, stratified_split, stratified_kfold, MinMaxScaler,
)
from src.ml.evaluator import evaluate, kfold_evaluate, precision_recall_f1

X, y, classes = load_crop_dataset('../data/raw/crop_recommendation.csv')
n_classes = len(classes)
X_train, X_test, y_train, y_test = stratified_split(X, y, test_size=0.2, seed=42)

scaler = MinMaxScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

X_train.shape, X_test.shape, n_classes

((1760, 7), (440, 7), 22)

## 1. Búsqueda de hiperparámetros con k-fold (sobre train escalado)

KNN sobre `k ∈ {1, 3, 5, 7, 9, 11, 15}` y Decision Tree sobre `max_depth ∈ {5, 8, 10, 12, 15, None}`. Métrica de selección: F1 macro promedio sobre 5 folds del train. Los demás modelos usan defaults razonables.

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

knn_rows = []
for k in [1, 3, 5, 7, 9, 11, 15]:
    folds = stratified_kfold(y_train, n_splits=5, seed=42)
    res = kfold_evaluate(
        lambda k=k: KNeighborsClassifier(n_neighbors=k),
        X_train_s, y_train, n_classes, folds,
    )
    knn_rows.append({'k': k, **{m: round(v, 4) for m, v in res.items()}})
knn_df = pd.DataFrame(knn_rows)
knn_best_k = int(knn_df.loc[knn_df['f1_macro_mean'].idxmax(), 'k'])
knn_df

,k,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,1,0.9756,0.0064,0.9754,0.0064
1,3,0.9784,0.0050,0.9782,0.0054
2,5,0.9790,0.0058,0.9786,0.0063
3,7,0.9767,0.0066,0.9766,0.0067
4,9,0.9722,0.0105,0.9720,0.0108
5,11,0.9710,0.0099,0.9706,0.0103
6,15,0.9648,0.0161,0.9642,0.0167


In [3]:
dt_rows = []
for depth in [5, 8, 10, 12, 15, None]:
    folds = stratified_kfold(y_train, n_splits=5, seed=42)
    res = kfold_evaluate(
        lambda d=depth: DecisionTreeClassifier(max_depth=d, random_state=42),
        X_train_s, y_train, n_classes, folds,
    )
    dt_rows.append({'max_depth': str(depth), **{m: round(v, 4) for m, v in res.items()}})
dt_df = pd.DataFrame(dt_rows)
dt_best_depth_str = dt_df.loc[dt_df['f1_macro_mean'].idxmax(), 'max_depth']
dt_best_depth = None if dt_best_depth_str == 'None' else int(dt_best_depth_str)
dt_df

,max_depth,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,5,0.4085,0.0011,0.3125,0.0102
1,8,0.8739,0.0272,0.8491,0.0366
2,10,0.9727,0.0109,0.9731,0.0106
3,12,0.9830,0.0051,0.9829,0.0052
4,15,0.9830,0.0074,0.9830,0.0075
5,None,0.9830,0.0074,0.9830,0.0075


In [4]:
print(f'KNN: k* = {knn_best_k}')
print(f'DT:  max_depth* = {dt_best_depth_str}')

KNN: k* = 5
DT:  max_depth* = 15


## 2. Entrenamiento final y evaluación sobre test

In [5]:
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

models = {
    'KNN':           KNeighborsClassifier(n_neighbors=knn_best_k),
    'GaussianNB':    GaussianNB(),
    'DecisionTree':  DecisionTreeClassifier(max_depth=dt_best_depth, random_state=42),
    'SVM_RBF':       SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'MLP':           MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

results = {}
for name, m in models.items():
    results[name] = evaluate(m, X_train_s, y_train, X_test_s, y_test, n_classes)

summary = pd.DataFrame({
    name: {
        'accuracy':   round(r.accuracy, 4),
        'f1_macro':   round(r.f1_macro, 4),
        'fit_time_s': round(r.fit_time_sec, 3),
        'pred_time_s': round(r.predict_time_sec, 3),
    } for name, r in results.items()
}).T.sort_values('f1_macro', ascending=False)
summary

,accuracy,f1_macro,fit_time_s,pred_time_s
GaussianNB,1.0000,1.0000,0.003,0.001
SVM_RBF,0.9864,0.9863,0.050,0.030
KNN,0.9818,0.9817,0.002,0.017
DecisionTree,0.9795,0.9796,0.012,0.000
MLP,0.9773,0.9768,12.904,0.004


## 3. Selección del modelo ganador

Criterio: mayor F1 macro en test. Como desempate (si la diferencia es ≤ 0.005) se prefiere el modelo con menor tiempo de entrenamiento.

In [6]:
ranked = summary.copy()
top_f1 = ranked['f1_macro'].max()
candidates = ranked[ranked['f1_macro'] >= top_f1 - 0.005]
winner_name = candidates.sort_values('fit_time_s').index[0]
print(f'Modelo ganador: {winner_name}')
print(f'F1 macro: {results[winner_name].f1_macro:.4f}')
print(f'Accuracy: {results[winner_name].accuracy:.4f}')
print(f'Empatados (delta f1 <= 0.005): {list(candidates.index)}')

Modelo ganador: GaussianNB
F1 macro: 1.0000
Accuracy: 1.0000
Empatados (delta f1 <= 0.005): ['GaussianNB']


## 4. Matriz de confusión del ganador

In [7]:
cm = results[winner_name].confusion
off_diag = cm.sum() - np.trace(cm)
print(f'Errores totales en test: {off_diag} / {cm.sum()}')
errors = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            errors.append({'verdadera': classes[i], 'predicha': classes[j], 'n': int(cm[i, j])})
pd.DataFrame(errors).sort_values('n', ascending=False).head(10) if errors else 'sin errores'

Errores totales en test: 0 / 440


'sin errores'

## 5. Implementación propia del ganador (`src/ml/models/champion.py`)

Esta sección se completa después de fijar el ganador. Carga `Champion` desde `src/ml/models/champion.py`, entrena con el mismo split y escalado, y compara F1 macro contra la versión sklearn. Criterio de validación: diferencia absoluta ≤ 0.03.

In [8]:
from src.ml.models.champion import Champion

champion = Champion()
champion_metrics = evaluate(champion, X_train_s, y_train, X_test_s, y_test, n_classes)

sklearn_f1 = results[winner_name].f1_macro
own_f1 = champion_metrics.f1_macro
diff = abs(sklearn_f1 - own_f1)

pd.DataFrame({
    f'sklearn ({winner_name})': {'accuracy': round(results[winner_name].accuracy, 4), 'f1_macro': round(sklearn_f1, 4)},
    'Champion (propio)':         {'accuracy': round(champion_metrics.accuracy, 4),    'f1_macro': round(own_f1, 4)},
}).T.assign(**{'diff_f1': round(diff, 4), 'validacion (<=0.03)': bool(diff <= 0.03)})

,accuracy,f1_macro,diff_f1,validacion (<=0.03)
sklearn (GaussianNB),1.0,1.0,0.0,True
Champion (propio),1.0,1.0,0.0,True
